this is !!!!! EXplain koi tum

In [ ]:

%pip install -q "mediapipe>=0.10.20,<0.11" opencv-python-headless numpy matplotlib


In [ ]:

from google.colab import files

uploaded = files.upload()

video_path = next(iter(uploaded))

print("Video uploaded successfully!")
print("Filename:", video_path)


In [ ]:

from IPython.display import display, Video

display(Video(video_path, embed=True, width=640))


In [ ]:

# Step 6: Detect ONLY the patient's LEFT leg

import cv2
import mediapipe as mp
import matplotlib.pyplot as plt
import urllib.request

# Download MediaPipe pose model
model_url = (
    "https://storage.googleapis.com/mediapipe-models/"
    "pose_landmarker/pose_landmarker_lite/float16/1/"
    "pose_landmarker_lite.task"
)

urllib.request.urlretrieve(
    model_url,
    "pose_landmarker.task"
)

# Read video frame
cap = cv2.VideoCapture(video_path)

total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.set(cv2.CAP_PROP_POS_FRAMES, total_frames // 2)

success, frame = cap.read()
cap.release()

if not success:
    raise ValueError("Could not read video frame.")

rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

# Initialize MediaPipe
options = mp.tasks.vision.PoseLandmarkerOptions(
    base_options=mp.tasks.BaseOptions(
        model_asset_path="pose_landmarker.task"
    ),
    running_mode=mp.tasks.vision.RunningMode.IMAGE
)

with mp.tasks.vision.PoseLandmarker.create_from_options(
    options
) as detector:

    image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=rgb
    )

    result = detector.detect(image)

# Visualize LEFT LEG only
plt.figure(figsize=(8, 8))
plt.imshow(rgb)

if result.pose_landmarks:

    landmarks = result.pose_landmarks[0]
    h, w = rgb.shape[:2]

    left_leg = {
        23: "Left Hip",
        25: "Left Knee",
        27: "Left Ankle"
    }

    xs = []
    ys = []

    for idx, name in left_leg.items():

        point = landmarks[idx]

        x = point.x * w
        y = point.y * h

        xs.append(x)
        ys.append(y)

        plt.scatter(x, y, color="red", s=90)
        plt.text(x + 10, y, name,
                 color="yellow", fontsize=11)

    # Connect hip -> knee -> ankle
    plt.plot(
        xs, ys,
        color="lime",
        linewidth=3
    )

    print("Patient's LEFT leg detected!")

else:
    print("No person detected.")

plt.axis("off")
plt.show()


In [ ]:

#Calculate projected knee deviation angle

import numpy as np

def calculate_knee_angle(hip, knee, ankle):
    hip = np.array([hip.x, hip.y])
    knee = np.array([knee.x, knee.y])
    ankle = np.array([ankle.x, ankle.y])

    # Vectors from knee to hip and ankle
    vector1 = hip - knee
    vector2 = ankle - knee

    # Calculate joint angle
    cosine = np.dot(vector1, vector2) / (
        np.linalg.norm(vector1) *
        np.linalg.norm(vector2)
    )

    cosine = np.clip(cosine, -1.0, 1.0)
    angle = np.degrees(np.arccos(cosine))

    # Deviation from straight (180 degrees)
    deviation = 180 - angle

    return angle, deviation

if result.pose_landmarks:
    landmarks = result.pose_landmarks[0]

    hip = landmarks[23]
    knee = landmarks[25]
    ankle = landmarks[27]

    angle, deviation = calculate_knee_angle(
        hip, knee, ankle
    )

    print(f"Joint angle: {angle:.2f} degrees")
    print(f"Projected deviation: {deviation:.2f} degrees")


In [ ]:

# Step 8: Analyze the entire squat video

import pandas as pd

cap = cv2.VideoCapture(video_path)

fps = cap.get(cv2.CAP_PROP_FPS)

if fps <= 0:
    raise ValueError("Cannot determine video FPS.")

angles = []
times = []

options = mp.tasks.vision.PoseLandmarkerOptions(
    base_options=mp.tasks.BaseOptions(
        model_asset_path="pose_landmarker.task"
    ),
    running_mode=mp.tasks.vision.RunningMode.VIDEO
)

with mp.tasks.vision.PoseLandmarker.create_from_options(
    options
) as detector:

    frame_number = 0

    while True:
        success, frame = cap.read()

        if not success:
            break

        rgb = cv2.cvtColor(
            frame, cv2.COLOR_BGR2RGB
        )

        image = mp.Image(
            image_format=mp.ImageFormat.SRGB,
            data=rgb
        )

        timestamp_ms = int(frame_number / fps * 1000)

        detection = detector.detect_for_video(
            image, timestamp_ms
        )

        if detection.pose_landmarks:
            landmarks = detection.pose_landmarks[0]

            angle, deviation = calculate_knee_angle(
                landmarks[23],
                landmarks[25],
                landmarks[27]
            )
        else:
            deviation = np.nan

        angles.append(deviation)
        times.append(frame_number / fps)

        frame_number += 1

cap.release()

df = pd.DataFrame({
    "Time": times,
    "Projected_Deviation": angles
})

plt.figure(figsize=(12, 5))

plt.plot(
    df["Time"],
    df["Projected_Deviation"]
)

plt.xlabel("Time (seconds)")
plt.ylabel("Projected Knee Deviation (degrees)")
plt.title("Left Knee Movement During Single-Leg Squat")
plt.grid(True)
plt.show()

print("Total frames analyzed:", len(df))
print("Valid measurements:", df["Projected_Deviation"].notna().sum())

In [ ]:

# Step 9: Corrected signed FPPA calculation
# Patient's anatomical LEFT leg only

import numpy as np

def calculate_fppa(hip, knee, ankle, width, height):

    # Convert normalized landmarks to pixel coordinates
    H = np.array([hip.x * width, hip.y * height])
    K = np.array([knee.x * width, knee.y * height])
    A = np.array([ankle.x * width, ankle.y * height])

    # Vectors from knee to hip and ankle
    v1 = H - K
    v2 = A - K

    # Check for degenerate vectors
    if np.linalg.norm(v1) < 1e-8 or np.linalg.norm(v2) < 1e-8:
        return np.nan

    # Signed angular deviation from straight alignment
    cross = v1[0] * v2[1] - v1[1] * v2[0]
    dot = np.dot(v1, v2)

    signed_angle = np.degrees(np.arctan2(cross, dot))

    # For a front-facing, non-mirrored image of the LEFT leg:
    # Positive = knee moves medially relative to hip-ankle
    # Negative = knee moves laterally
    fppa = signed_angle

    return fppa

print("Corrected FPPA function ready!")


In [ ]:

# Step 10: Calculate signed FPPA for every frame

import cv2
import mediapipe as mp
import matplotlib.pyplot as plt
import pandas as pd

cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS)

if fps <= 0:
    cap.release()
    raise ValueError("Invalid video frame rate.")

fppa_values = []
time_values = []

options = mp.tasks.vision.PoseLandmarkerOptions(
    base_options=mp.tasks.BaseOptions(
        model_asset_path="pose_landmarker.task"
    ),
    running_mode=mp.tasks.vision.RunningMode.VIDEO
)

with mp.tasks.vision.PoseLandmarker.create_from_options(
    options
) as detector:

    frame_number = 0

    while True:
        success, frame = cap.read()

        if not success:
            break

        height, width = frame.shape[:2]

        rgb = cv2.cvtColor(
            frame, cv2.COLOR_BGR2RGB
        )

        image = mp.Image(
            image_format=mp.ImageFormat.SRGB,
            data=rgb
        )

        timestamp_ms = int(frame_number / fps * 1000)

        detection = detector.detect_for_video(
            image,
            timestamp_ms
        )

        if detection.pose_landmarks:
            landmarks = detection.pose_landmarks[0]

            fppa = calculate_fppa(
                landmarks[23],
                landmarks[25],
                landmarks[27],
                width,
                height
            )
        else:
            fppa = np.nan

        fppa_values.append(fppa)
        time_values.append(frame_number / fps)
        frame_number += 1

cap.release()

# Save measurements
df_fppa = pd.DataFrame({
    "Time": time_values,
    "FPPA": fppa_values
})

# Plot the new graph
plt.figure(figsize=(12, 5))

plt.plot(
    df_fppa["Time"],
    df_fppa["FPPA"],
    color="blue",
    label="Signed FPPA"
)

plt.axhline(
    0,
    color="black",
    linestyle="--",
    linewidth=1
)

plt.xlabel("Time (seconds)")
plt.ylabel("Signed FPPA (degrees)")
plt.title("Left Knee FPPA During Single-Leg Squat")
plt.legend()
plt.grid(True)
plt.show()

print("Frames analyzed:", len(df_fppa))
print("Maximum FPPA:", round(df_fppa["FPPA"].max(), 2))
print("Minimum FPPA:", round(df_fppa["FPPA"].min(), 2))


In [ ]:

# Step 11: Correct the signed FPPA values
# This fixes the -180 degree offset

import numpy as np
import matplotlib.pyplot as plt

# Convert previously calculated signed angles
df_fppa["FPPA_Corrected"] = (
    df_fppa["FPPA"] + 180
)

# Display corrected graph
plt.figure(figsize=(12, 5))

plt.plot(
    df_fppa["Time"],
    df_fppa["FPPA_Corrected"],
    color="blue",
    linewidth=2,
    label="Corrected FPPA"
)

plt.axhline(
    y=0,
    color="black",
    linestyle="--",
    label="Straight alignment"
)

plt.xlabel("Time (seconds)")
plt.ylabel("FPPA (degrees)")
plt.title("Corrected Left Knee FPPA During Single-Leg Squat")

plt.legend()
plt.grid(True)
plt.show()

print("Maximum FPPA:",
      round(df_fppa["FPPA_Corrected"].max(), 2))

print("Minimum FPPA:",
      round(df_fppa["FPPA_Corrected"].min(), 2))


In [ ]:

# Step 12: Detect squat repetitions from hip movement

import cv2
import mediapipe as mp
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks, savgol_filter

cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS)

if fps <= 0:
    cap.release()
    raise ValueError("Invalid video FPS")

hip_heights = []
timestamps = []

options = mp.tasks.vision.PoseLandmarkerOptions(
    base_options=mp.tasks.BaseOptions(
        model_asset_path="pose_landmarker.task"
    ),
    running_mode=mp.tasks.vision.RunningMode.VIDEO
)

with mp.tasks.vision.PoseLandmarker.create_from_options(
    options
) as detector:

    frame_number = 0

    while True:
        success, frame = cap.read()
        if not success:
            break

        rgb = cv2.cvtColor(
            frame, cv2.COLOR_BGR2RGB
        )

        image = mp.Image(
            image_format=mp.ImageFormat.SRGB,
            data=rgb
        )

        timestamp_ms = int(frame_number / fps * 1000)

        result_video = detector.detect_for_video(
            image, timestamp_ms
        )

        if result_video.pose_landmarks:
            lm = result_video.pose_landmarks[0]

            # Left hip (23), left ankle (27)
            hip = lm[23]
            ankle = lm[27]

            # Normalized vertical hip-ankle distance
            hip_height = ankle.y - hip.y
        else:
            hip_height = np.nan

        hip_heights.append(hip_height)
        timestamps.append(frame_number / fps)
        frame_number += 1

cap.release()

# Fill missing measurements
heights = np.array(hip_heights, dtype=float)

if np.isnan(heights).all():
    raise ValueError("No hip landmarks detected")

valid = np.isfinite(heights)
heights = np.interp(
    np.arange(len(heights)),
    np.flatnonzero(valid),
    heights[valid]
)

# Smooth the signal
window = min(31, len(heights) if len(heights) % 2 else len(heights)-1)
if window < 5:
    raise ValueError("Video is too short")

smooth = savgol_filter(heights, window, 3)

# Bottom of squat corresponds approximately
# to local minima in hip-to-ankle vertical distance
prominence = max(0.015, np.ptp(smooth) * 0.15)

peaks, properties = find_peaks(
    -smooth,
    distance=max(1, int(fps * 1.0)),
    prominence=prominence
)

# Plot detected squat bottoms
plt.figure(figsize=(12, 5))
plt.plot(timestamps, smooth, label="Hip-ankle distance")
plt.scatter(
    np.array(timestamps)[peaks],
    smooth[peaks],
    color="red",
    s=80,
    label="Possible squat bottom"
)

plt.xlabel("Time (seconds)")
plt.ylabel("Normalized vertical distance")
plt.title("Single-Leg Squat Repetition Detection")
plt.legend()
plt.grid(True)
plt.show()

print("Possible squat repetitions detected:", len(peaks))
print("Squat bottom times:",
      np.round(np.array(timestamps)[peaks], 2))


In [ ]:

# Step 13: Compare knee FPPA with squat depth

import matplotlib.pyplot as plt
import numpy as np

# Use results from Steps 11 and 12
times = np.array(timestamps)

fppa = df_fppa["FPPA_Corrected"].to_numpy()

# Make sure the data has matching lengths
n = min(len(times), len(fppa), len(smooth))

times = times[:n]
fppa = fppa[:n]
hip_signal = smooth[:n]

fig, ax1 = plt.subplots(figsize=(13, 6))

# Knee angle
ax1.plot(
    times,
    fppa,
    color="blue",
    linewidth=2,
    label="Left Knee FPPA"
)

ax1.set_xlabel("Time (seconds)")
ax1.set_ylabel("FPPA (degrees)", color="blue")
ax1.tick_params(axis="y", labelcolor="blue")

# Hip movement on second axis
ax2 = ax1.twinx()

ax2.plot(
    times,
    hip_signal,
    color="orange",
    linewidth=2,
    label="Hip-Ankle Vertical Distance"
)

ax2.set_ylabel(
    "Normalized Hip-Ankle Distance",
    color="orange"
)

ax2.tick_params(axis="y", labelcolor="orange")

# Mark squat bottom
bottom_idx = int(np.argmin(hip_signal))
bottom_time = times[bottom_idx]

ax1.axvline(
    bottom_time,
    color="red",
    linestyle="--",
    label="Squat Bottom"
)

plt.title("Knee FPPA and Squat Depth Over Time")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()

ax1.legend(lines1 + lines2, labels1 + labels2)
ax1.grid(True, alpha=0.3)

plt.show()

print(f"Squat bottom: {bottom_time:.2f} seconds")
print(f"FPPA at squat bottom: {fppa[bottom_idx]:.2f} degrees")
print(f"Maximum measured FPPA: {np.nanmax(fppa):.2f} degrees")


In [ ]:

# Step 14: Inspect knee alignment at key moments

import cv2
import matplotlib.pyplot as plt
import numpy as np

# Use measurements from previous steps
inspection_times = [
    0.5,
    float(bottom_time),
    float(df_fppa.loc[
        df_fppa["FPPA_Corrected"].idxmax(), "Time"
    ])
]

titles = [
    "Start of Squat",
    "Bottom of Squat",
    "Maximum Projected Deviation"
]

cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS)

fig, axes = plt.subplots(1, 3, figsize=(16, 6))

options = mp.tasks.vision.PoseLandmarkerOptions(
    base_options=mp.tasks.BaseOptions(
        model_asset_path="pose_landmarker.task"
    ),
    running_mode=mp.tasks.vision.RunningMode.IMAGE
)

with mp.tasks.vision.PoseLandmarker.create_from_options(
    options
) as detector:

    for ax, t, title in zip(
        axes, inspection_times, titles
    ):

        cap.set(
            cv2.CAP_PROP_POS_MSEC,
            t * 1000
        )

        success, frame = cap.read()

        if not success:
            ax.set_title("Frame unavailable")
            ax.axis("off")
            continue

        rgb = cv2.cvtColor(
            frame, cv2.COLOR_BGR2RGB
        )

        image = mp.Image(
            image_format=mp.ImageFormat.SRGB,
            data=rgb
        )

        detection = detector.detect(image)
        ax.imshow(rgb)

        if detection.pose_landmarks:
            lm = detection.pose_landmarks[0]
            h, w = rgb.shape[:2]

            # Patient's LEFT leg
            points = np.array([
                [lm[i].x * w, lm[i].y * h]
                for i in [23, 25, 27]
            ])

            ax.plot(
                points[:, 0],
                points[:, 1],
                color="lime",
                linewidth=3
            )

            ax.scatter(
                points[:, 0],
                points[:, 1],
                color="red",
                s=60
            )

            # Reference line from hip to ankle
            ax.plot(
                [points[0, 0], points[2, 0]],
                [points[0, 1], points[2, 1]],
                color="yellow",
                linestyle="--",
                linewidth=2
            )

        ax.set_title(f"{title}\nTime: {t:.2f}s")
        ax.axis("off")

cap.release()
plt.tight_layout()
plt.show()


In [ ]:

# Step 15: Calculate directional knee displacement
# Anatomical LEFT leg, front-facing non-mirrored video

import numpy as np

def knee_medial_displacement(hip, knee, ankle, width, height):

    H = np.array([hip.x * width, hip.y * height])
    K = np.array([knee.x * width, knee.y * height])
    A = np.array([ankle.x * width, ankle.y * height])

    # Hip-to-ankle reference at the knee's vertical level
    vertical_span = A[1] - H[1]

    if abs(vertical_span) < 1e-8:
        return np.nan

    fraction = (K[1] - H[1]) / vertical_span
    reference_x = H[0] + fraction * (A[0] - H[0])

    # Patient's left is image-right when facing the camera.
    # Medial movement therefore points image-left.
    medial_displacement_px = reference_x - K[0]

    # Normalize by projected hip-to-ankle length
    leg_length = np.linalg.norm(A - H)

    if leg_length < 1e-8:
        return np.nan

    return medial_displacement_px / leg_length

print("Directional knee displacement function ready!")


In [ ]:

# Step 16: Verify knee movement direction

cap = cv2.VideoCapture(video_path)

with mp.tasks.vision.PoseLandmarker.create_from_options(
    options
) as detector:

    for t, title in zip(inspection_times, titles):

        cap.set(cv2.CAP_PROP_POS_MSEC, t * 1000)
        success, frame = cap.read()

        if not success:
            continue

        height, width = frame.shape[:2]
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        image = mp.Image(
            image_format=mp.ImageFormat.SRGB,
            data=rgb
        )

        detection = detector.detect(image)

        if detection.pose_landmarks:
            lm = detection.pose_landmarks[0]

            displacement = knee_medial_displacement(
                lm[23], lm[25], lm[27],
                width, height
            )

            direction = (
                "Medial (valgus direction)"
                if displacement > 0
                else "Lateral (varus direction)"
                if displacement < 0
                else "Aligned"
            )

            print(f"{title} ({t:.2f}s)")
            print(f"  Displacement: {displacement:.4f}")
            print(f"  Direction: {direction}")
            print()

cap.release()


In [ ]:

# Step 17: Create an annotated squat video

import cv2
import mediapipe as mp
import numpy as np

cap = cv2.VideoCapture(video_path)

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

output_path = "annotated_knee_video.mp4"

writer = cv2.VideoWriter(
    output_path,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (width, height)
)

options_video = mp.tasks.vision.PoseLandmarkerOptions(
    base_options=mp.tasks.BaseOptions(
        model_asset_path="pose_landmarker.task"
    ),
    running_mode=mp.tasks.vision.RunningMode.VIDEO
)

frame_number = 0

with mp.tasks.vision.PoseLandmarker.create_from_options(
    options_video
) as detector:

    while True:
        success, frame = cap.read()

        if not success:
            break

        rgb = cv2.cvtColor(
            frame, cv2.COLOR_BGR2RGB
        )

        image = mp.Image(
            image_format=mp.ImageFormat.SRGB,
            data=rgb
        )

        timestamp_ms = int(frame_number / fps * 1000)

        result = detector.detect_for_video(
            image, timestamp_ms
        )

        if result.pose_landmarks:

            lm = result.pose_landmarks[0]

            points = [
                (int(lm[i].x * width),
                 int(lm[i].y * height))
                for i in [23, 25, 27]
            ]

            hip, knee, ankle = points

            displacement = knee_medial_displacement(
                lm[23], lm[25], lm[27],
                width, height
            )

            # Prototype visualization tolerance
            if not np.isfinite(displacement):
                color = (160, 160, 160)
                direction = "Unavailable"
            elif displacement > 0.01:
                color = (0, 0, 255)
                direction = "Medial"
            elif displacement < -0.01:
                color = (255, 0, 0)
                direction = "Lateral"
            else:
                color = (0, 255, 0)
                direction = "Near reference"

            # Draw leg
            cv2.line(frame, hip, knee, color, 4)
            cv2.line(frame, knee, ankle, color, 4)

            for point in points:
                cv2.circle(frame, point, 7, color, -1)

            # Draw hip-ankle reference line
            cv2.line(
                frame, hip, ankle,
                (0, 255, 255), 2
            )

            # Display movement information
            cv2.putText(
                frame,
                f"Direction: {direction}",
                (25, 45),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                color,
                2
            )

            cv2.putText(
                frame,
                f"Displacement: {displacement:.3f}",
                (25, 85),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (255, 255, 255),
                2
            )

        writer.write(frame)
        frame_number += 1

cap.release()
writer.release()

print("Annotated video created:", output_path)
print("Frames processed:", frame_number)


In [ ]:

# Step 18: Display the annotated video

from IPython.display import Video, display

display(Video(
    "annotated_knee_video.mp4",
    embed=True,
    width=700
))


In [ ]:

%pip install -q gradio


In [ ]:

import gradio as gr

print("Gradio installed successfully!")
print("Gradio version:", gr.__version__)


In [ ]:

# STEP 19B - Knee Movement Analysis Gradio App

import gradio as gr
import cv2
import mediapipe as mp
import numpy as np
import tempfile
import os
import subprocess

def analyze_squat(video):

    if video is None:
        raise gr.Error("Please upload a video first.")

    cap = cv2.VideoCapture(video)

    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    if fps <= 0 or width <= 0 or height <= 0:
        cap.release()
        raise gr.Error("Unable to read the video.")

    # Temporary output files
    temp_dir = tempfile.mkdtemp()
    raw_output = os.path.join(temp_dir, "raw.mp4")
    final_output = os.path.join(temp_dir, "annotated.mp4")

    writer = cv2.VideoWriter(
        raw_output,
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps,
        (width, height)
    )

    if not writer.isOpened():
        cap.release()
        raise gr.Error("Could not create output video.")

    measurements = []
    frame_number = 0

    options = mp.tasks.vision.PoseLandmarkerOptions(
        base_options=mp.tasks.BaseOptions(
            model_asset_path="pose_landmarker.task"
        ),
        running_mode=mp.tasks.vision.RunningMode.VIDEO
    )

    try:
        with mp.tasks.vision.PoseLandmarker.create_from_options(
            options
        ) as detector:

            while True:
                success, frame = cap.read()
                if not success:
                    break

                rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                image = mp.Image(
                    image_format=mp.ImageFormat.SRGB,
                    data=rgb
                )

                timestamp_ms = int(frame_number / fps * 1000)

                detection = detector.detect_for_video(
                    image, timestamp_ms
                )

                if detection.pose_landmarks:
                    lm = detection.pose_landmarks[0]

                    points = [
                        (
                            int(lm[i].x * width),
                            int(lm[i].y * height)
                        )
                        for i in [23, 25, 27]
                    ]

                    hip, knee, ankle = points

                    # Uses your function from Step 15
                    displacement = knee_medial_displacement(
                        lm[23], lm[25], lm[27],
                        width, height
                    )

                    measurements.append(displacement)

                    if not np.isfinite(displacement):
                        color = (160, 160, 160)
                        direction = "Unavailable"
                    elif displacement > 0.01:
                        color = (0, 0, 255)
                        direction = "Medial"
                    elif displacement < -0.01:
                        color = (255, 0, 0)
                        direction = "Lateral"
                    else:
                        color = (0, 255, 0)
                        direction = "Near reference"

                    cv2.line(frame, hip, knee, color, 4)
                    cv2.line(frame, knee, ankle, color, 4)
                    cv2.line(
                        frame, hip, ankle,
                        (0, 255, 255), 2
                    )

                    for point in points:
                        cv2.circle(frame, point, 7, color, -1)

                    cv2.putText(
                        frame,
                        f"Direction: {direction}",
                        (25, 45),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.8,
                        color,
                        2
                    )

                else:
                    measurements.append(np.nan)

                writer.write(frame)
                frame_number += 1

    finally:
        cap.release()
        writer.release()

    # Convert to browser-compatible H.264 video
    subprocess.run(
        [
            "ffmpeg", "-y",
            "-loglevel", "error",
            "-i", raw_output,
            "-c:v", "libx264",
            "-pix_fmt", "yuv420p",
            "-movflags", "+faststart",
            "-an", final_output
        ],
        check=True
    )

    valid = np.array(measurements, dtype=float)
    valid = valid[np.isfinite(valid)]

    if len(valid) == 0:
        summary = "No valid body landmarks detected."
    else:
        summary = (
            f"Frames analyzed: {frame_number}\n"
            f"Maximum displacement: {valid.max():.4f}\n"
            f"Minimum displacement: {valid.min():.4f}\n"
            f"Average displacement: {valid.mean():.4f}"
        )

    return final_output, summary


# Build the Gradio interface
demo = gr.Interface(
    fn=analyze_squat,
    inputs=gr.Video(
        label="Upload Single-Leg Squat Video"
    ),
    outputs=[
        gr.Video(label="Annotated Knee Tracking Video"),
        gr.Textbox(label="Knee Movement Results")
    ],
    title="AI-Based Knee Movement Analysis",
    description=(
        "Upload a front-facing single-leg squat video. "
        "This prototype tracks the patient's anatomical "
        "left hip, knee, and ankle using MediaPipe. "
        "Movement direction is experimental and is "
        "not a medical diagnosis."
    )
)

# Launch application
demo.launch(share=True, debug=True)
